<a href="https://colab.research.google.com/github/2anizirong/AI-Application/blob/whisper_learn/whisper_complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tarfile
import os
import shutil
from tqdm import tqdm

# 설정
tar_path = "/content/drive/MyDrive/whisper_data/TS_kor_free_01.tar"
extract_dir = "/content/audio_subset"
os.makedirs(extract_dir, exist_ok=True)

# 추출 개수 설정
MAX_FILES = 100000
extracted = 0

with tarfile.open(tar_path, "r") as tar:
    print(f" {MAX_FILES}개의 .wav 파일만 추출합니다...\n")
    for member in tqdm(tar, desc="파일 검색 중"):
        # .wav 파일만 필터링
        if member.isfile() and member.name.endswith(".wav") and "__MACOSX" not in member.name:
            filename = os.path.basename(member.name)
            target_path = os.path.join(extract_dir, filename)
            with tar.extractfile(member) as src, open(target_path, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            extracted += 1
            if extracted >= MAX_FILES:
                break

print(f"\n 추출 완료! 총 {extracted}개의 .wav 파일이 {extract_dir}에 저장되었습니다.")

 100000개의 .wav 파일만 추출합니다...



파일 검색 중: 100181it [07:20, 227.68it/s]


 추출 완료! 총 100000개의 .wav 파일이 /content/audio_subset에 저장되었습니다.


In [ ]:
import os
import tarfile
from tqdm import tqdm
import shutil

# 경로 설정
tar_path = "/content/drive/MyDrive/whisper_data/TL_kor_free_01.tar"
extract_dir = "/content/labels"

# 디렉토리 준비
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)
os.makedirs(extract_dir, exist_ok=True)

# 압축 해제 시작
with tarfile.open(tar_path, "r") as tar:
    members = [m for m in tar.getmembers() if m.isfile()]ㄴ
    print(f" 총 {len(members)}개 파일 압축 해제 중...")
    with tqdm(total=len(members), desc="압축 해제 중", unit="file") as pbar:
        for member in members:
            tar.extract(member, path=extract_dir)
            pbar.update(1)

print(f"\n 라벨 데이터 전체 해제 완료 → {extract_dir}")

 총 404375개 파일 압축 해제 중...


압축 해제 중: 100%|██████████| 404375/404375 [00:34<00:00, 11858.65file/s]


 라벨 데이터 전체 해제 완료 → /content/labels


In [ ]:
import os
import json
from glob import glob

label_dir = "/content/labels"
audio_dir = "/content/audio_subset"

# 실제 오디오 파일 이름들 (집합으로 저장)
audio_files = set(os.listdir(audio_dir))

paired_data = []

json_files = glob(os.path.join(label_dir, "**/*.json"), recursive=True)

for json_path in json_files:
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            file_name = data.get("File", {}).get("FileName")  # ex: K0001...wav
            transcription = data.get("Transcription", {}).get("LabelText")
            if file_name in audio_files and transcription:
                paired_data.append({
                    "audio": os.path.join(audio_dir, file_name),
                    "text": transcription.strip()
                })
    except Exception as e:
        continue  # 깨진 파일은 무시

print(f" 매칭된 오디오-텍스트 쌍 수: {len(paired_data)}개")

 매칭된 오디오-텍스트 쌍 수: 100000개


In [ ]:
!pip install datasets

In [ ]:
import os
import json
from glob import glob
from datasets import Dataset

label_dir = "/content/labels"
audio_dir = "/content/audio_subset"

audio_files = set(os.listdir(audio_dir))
paired_data = []

json_files = glob(os.path.join(label_dir, "**/*.json"), recursive=True)

for json_path in json_files:
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            file_name = data.get("File", {}).get("FileName")
            transcription = data.get("Transcription", {}).get("LabelText")
            if file_name in audio_files and transcription:
                paired_data.append({
                    "audio": os.path.join(audio_dir, file_name),
                    "text": transcription.strip()
                })
    except Exception:
        continue

# Hugging Face Dataset으로 변환
dataset = Dataset.from_list(paired_data)
print(f" Whisper 학습용 데이터셋 생성 완료: {len(dataset)}개 샘플")
dataset.shuffle(seed=42).select(range(5))

 Whisper 학습용 데이터셋 생성 완료: 100000개 샘플


Dataset({
    features: ['audio', 'text'],
    num_rows: 5
})

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# 모델과 프로세서 로드
model_name = "openai/whisper-tiny"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.75k [00:00<?, ?B/s]

In [ ]:
import torchaudio

In [ ]:
!pip install --no-cache-dir accelerate==0.24.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.4/261.4 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 369.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 319.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 191.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 204.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 162.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 88.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    U

In [ ]:
from datasets import load_from_disk, concatenate_datasets
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

dataset_paths = [f"/content/drive/MyDrive/whisper_cached_part_{i}" for i in range(10)]

def load_one(path):
    return load_from_disk(path)

loaded_datasets = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(load_one, path): path for path in dataset_paths}
    for future in tqdm(as_completed(futures), total=len(futures), desc="데이터 로드 중"):
        try:
            dataset = future.result()
            loaded_datasets.append(dataset)
        except Exception as e:
            print(f"{futures[future]} 로딩 실패: {e}")

# 병합 및 저장
train_dataset = concatenate_datasets(loaded_datasets)
train_dataset.save_to_disk("/content/drive/MyDrive/whisper_combined")

print("\n병합 완료 및 whisper_combined 폴더로 저장됨.")


데이터 로드 중: 100%|██████████| 10/10 [00:00<00:00, 20420.18it/s]

/content/drive/MyDrive/whisper_cached_part_2 로딩 실패: Directory /content/drive/MyDrive/whisper_cached_part_2 not found
/content/drive/MyDrive/whisper_cached_part_8 로딩 실패: Directory /content/drive/MyDrive/whisper_cached_part_8 not found
/content/drive/MyDrive/whisper_cached_part_7 로딩 실패: Directory /content/drive/MyDrive/whisper_cached_part_7 not found
/content/drive/MyDrive/whisper_cached_part_6 로딩 실패: Directory /content/drive/MyDrive/whisper_cached_part_6 not found
/content/drive/MyDrive/whisper_cached_part_5 로딩 실패: Directory /content/drive/MyDrive/whisper_cached_part_5 not found
/content/drive/MyDrive/whisper_cached_part_4 로딩 실패: Directory /content/drive/MyDrive/whisper_cached_part_4 not found
/content/drive/MyDrive/whisper_cached_part_1 로딩 실패: Directory /content/drive/MyDrive/whisper_cached_part_1 not found
/content/drive/MyDrive/whisper_cached_part_0 로딩 실패: Directory /content/drive/MyDrive/whisper_cached_part_0 not found
/content/drive/MyDrive/whisper_cached_part_3 로딩 실패: Directory /c

ValueError: Unable to concatenate an empty list of datasets.

In [ ]:
from datasets import load_from_disk

# 저장된 병합 데이터를 다시 불러옴
train_dataset = load_from_disk("/content/drive/MyDrive/whisper_combined")

print("whisper_combined에서 train_dataset 로드 완료:", len(train_dataset), "샘플")

whisper_combined에서 train_dataset 로드 완료: 50000 샘플


/usr/local/lib/python3.11/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model_name = "openai/whisper-tiny"  # 필요시 'base', 'small', 'medium'으로 교체
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

# 언어 및 태스크 설정 (한국어 + 음성 텍스트 변환)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="ko", task="transcribe")

In [ ]:
!pip install --upgrade --no-cache-dir transformers==4.35.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 243.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 295.9 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.1
    Uninstalling tokenizers-0.21.1:
      Successfully uninstalled tokenizers-0.21.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.52.3
    Uninstalling transformers-4.52.3:
      Successfully uninstalled transformers-4.52.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.35.2 which is incompatible.


In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class CustomDataCollator:
    processor: Any
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(labels_batch["input_ids"] == self.processor.tokenizer.pad_token_id, -100)
        batch["labels"] = labels
        return batch

In [ ]:
# Huggingface Dataset 객체에서 상위 1만 개만 선택
mini_train_dataset = train_dataset.select(range(10000))

In [ ]:
# 학습에 사용하지 않은 다음 1만 개 데이터셋
next_10k_dataset = train_dataset.select(range(10000, 20000))

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# TrainingArguments → Seq2SeqTrainingArguments 그대로 사용해도 무방
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/whisper_finetuned_model",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    num_train_epochs=5,
    fp16=True,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
)

# Whisper 전용 Data Collator 설정
data_collator = CustomDataCollator(processor=processor)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=next_10k_dataset,
    data_collator=data_collator,
)

trainer.train()


Step,Training Loss
50,0.616200
100,0.612700
150,0.596300
200,0.601700
250,0.563100
300,0.575200
350,0.538000
400,0.542400
450,0.541500
500,0.503300


Step,Training Loss
50,0.616200
100,0.612700
150,0.596300
200,0.601700
250,0.563100
300,0.575200
350,0.538000
400,0.542400
450,0.541500
500,0.503300


TrainOutput(global_step=3125, training_loss=0.32959251647949217, metrics={'train_runtime': 7977.9452, 'train_samples_per_second': 6.267, 'train_steps_per_second': 0.392, 'total_flos': 1.230944256e+18, 'train_loss': 0.32959251647949217, 'epoch': 5.0})

In [ ]:
model.save_pretrained("/content/drive/MyDrive/whisper_finetuned_model")
processor.save_pretrained("/content/drive/MyDrive/whisper_finetuned_model")

[]

In [ ]:
import evaluate
from tqdm import tqdm
import torch
import re

wer_metric = evaluate.load("wer")

def trim_to_first_sentence(text):
    match = re.search(r'[.?!]', text)
    if match:
        return text[:match.end()]
    return text

unseen_dataset = train_dataset.select(range(23000, 50000))
test_subset = unseen_dataset.select(range(10))

references = []
predictions = []

for sample in tqdm(test_subset, desc="모델 추론 중"):
    input_features = torch.tensor(sample["input_features"]).unsqueeze(0).float().to(model.device)

    with torch.no_grad():
        pred_ids = model.generate(
            input_features,
            max_length=128,
            no_repeat_ngram_size=3,
            repetition_penalty=2.0,
            length_penalty=1.0,
            num_beams=5,
            early_stopping=True
        )

    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
    pred_text = trim_to_first_sentence(pred_text)
    predictions.append(pred_text)

    label_ids = sample["labels"]
    label_ids = [id if id != -100 else processor.tokenizer.pad_token_id for id in label_ids]
    ref_text = processor.tokenizer.decode(label_ids, skip_special_tokens=True)
    references.append(ref_text)

    print("정답:", ref_text)
    print("예측:", pred_text)
    print("———")

wer = wer_metric.compute(predictions=predictions, references=references)
print(f"\n전체 WER (Word Error Rate): {wer:.2%}")


모델 추론 중:  10%|█         | 1/10 [00:00<00:03,  2.44it/s]

🎧 정답: 색이 너무 어둡고 뭔가 깔끔하지 않은 것 같다.
🗣️ 예측:  색이 너무 어두고 뭔가 깔끔하지 않은 것 같다.
———


모델 추론 중:  20%|██        | 2/10 [00:00<00:02,  2.69it/s]

🎧 정답: 요즘은 영상편집에 관심이 많다.
🗣️ 예측:  요즘은 영상편집에 관심이 많다.
———


모델 추론 중:  30%|███       | 3/10 [00:02<00:06,  1.11it/s]

...

🎧 정답: 이 닦기도 너무 귀찮다.
🗣️ 예측: 이 닦기도 너무 귀찮다.
———


모델 추론 중:  90%|█████████ | 9/10 [00:05<00:00,  1.97it/s]

🎧 정답: 나는 조선의 왕들을 보면서 좀 이상했다.
🗣️ 예측:  나는 조선에 왕드를 보면서 좀 이상했다.
———


모델 추론 중: 100%|██████████| 10/10 [00:05<00:00,  1.78it/s]

🎧 정답: 나와 같은 취미를 가진 친구들이 많다.
🗣️ 예측:  나와 같은 TV를 가진 친구들이 많다.
———

✅ 전체 WER (Word Error Rate): 18.97%
